# Width Metrics Testing Notebook

**Project:** Pointcloud Analysis for Pedestrian Access  
**Component:** Width Metrics / Analysis  
**Author:** Sujeeth Gunasekaran

This notebook contains the full testing workflow for:

- Point-based sidewalk width estimation
- Boundary extraction testing
- Boundary-based width estimation
- Width filtering and validation
- Point-based vs boundary-based comparison
- Final integrated workflow testing

The notebook supports:

- `.laz`
- `.las`
- `.obj`

inputs for both classified point clouds and extracted boundary files.


## 1. Setup

This notebook assumes the following structure:

### Classified point cloud
- `classified/utrecht_mlp_classified.laz`

### Boundary extraction outputs
- `outputs/sidewalk_boundary_ALL.laz`
- `outputs/sidewalk_KI.laz`
- `outputs/sidewalk_HFE.laz`

### Width metrics output directory
- `outputs/width_metrics`


In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

# Main classified point cloud
INPUT_FILE = Path("classified/utrecht_mlp_classified.laz")

# Final separated boundary files
KERB_FILE = Path("outputs/sidewalk_KI.laz")
HFE_FILE = Path("outputs/sidewalk_HFE.laz")

# Combined extracted boundary file
BOUNDARY_ALL_FILE = Path("outputs/sidewalk_boundary_ALL.laz")

# Output directory
OUTPUT_DIR = Path("outputs/width_metrics")

# Output files
SEGMENT_CSV = OUTPUT_DIR / "sidewalk_segment_metrics.csv"
POINT_SUMMARY_JSON = OUTPUT_DIR / "sidewalk_metrics_summary.json"
BOUNDARY_SUMMARY_JSON = OUTPUT_DIR / "boundary_width_summary.json"

print("=== Input Files ===")
print("Classified file:", INPUT_FILE)
print("Exists:", INPUT_FILE.exists())

print("\nCombined boundary file:", BOUNDARY_ALL_FILE)
print("Exists:", BOUNDARY_ALL_FILE.exists())

print("\nKerb boundary file:", KERB_FILE)
print("Exists:", KERB_FILE.exists())

print("\nHFE boundary file:", HFE_FILE)
print("Exists:", HFE_FILE.exists())

print("\nOutput directory:", OUTPUT_DIR)


## 2. Run Point-Based Width Metrics

This tests the PCA-based sidewalk width estimation workflow.

The point-based workflow:
- uses classified sidewalk points
- estimates walking direction using PCA
- measures widths perpendicular to the dominant direction
- uses percentile filtering
- removes unrealistic segments
- estimates usable pedestrian width


In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

command = [
    "python3", "-m", "metrics.width_metrics",
    "--input", str(INPUT_FILE),
    "--output", str(OUTPUT_DIR),
]

result = subprocess.run(command, capture_output=True, text=True)

print(result.stdout)

if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Point-based workflow failed.")


## 3. View Point-Based Summary


In [ ]:
with open(POINT_SUMMARY_JSON, "r", encoding="utf-8") as file:
    point_summary = json.load(file)

point_summary


## 4. Load Segment-Level CSV Results


In [ ]:
df = pd.read_csv(SEGMENT_CSV)
df.head(10)


## 5. Segment Statistics


In [ ]:
df.describe()


## 6. Plot Overall Width vs Usable Width


In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(df["segment_id"], df["overall_width_m"], label="Overall Width")
plt.plot(df["segment_id"], df["usable_width_m"], label="Usable Width")

plt.xlabel("Segment ID")
plt.ylabel("Width (m)")
plt.title("Point-Based Sidewalk Width Metrics")
plt.legend()
plt.grid(True)

plt.show()


## 7. Boundary Extraction Validation

The current workflow also produces:

- `sidewalk_boundary_ALL.laz`
- separated kerb-side boundary files
- separated frontage/HFE-side boundary files

The combined boundary file can be opened in CloudCompare for visual validation.

The separated boundaries are used directly for the boundary-based width workflow.


In [ ]:
print("Combined boundary file exists:", BOUNDARY_ALL_FILE.exists())
print("Kerb boundary exists:", KERB_FILE.exists())
print("HFE boundary exists:", HFE_FILE.exists())


## 8. Run Boundary-Based Width Metrics

This workflow measures distances between:
- kerb-side boundaries
- frontage/HFE-side boundaries

The metrics module supports:
- `.obj`
- `.las`
- `.laz`

boundary files directly.


In [ ]:
command = [
    "python3", "-m", "metrics.width_metrics",
    "--kerb-file", str(KERB_FILE),
    "--hfe-file", str(HFE_FILE),
    "--output", str(OUTPUT_DIR),
]

result = subprocess.run(command, capture_output=True, text=True)

print(result.stdout)

if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Boundary-based workflow failed.")


## 9. View Boundary-Based Summary

The boundary workflow reports:
- raw widths
- filtered widths
- removed outlier counts
- median widths
- maximum widths


In [ ]:
with open(BOUNDARY_SUMMARY_JSON, "r", encoding="utf-8") as file:
    boundary_summary = json.load(file)

boundary_summary


## 10. Compare Point-Based and Boundary-Based Results


In [ ]:
comparison = pd.DataFrame([
    {
        "method": "Point-based PCA",
        "average_width_m": point_summary.get("average_overall_width_m"),
        "median_width_m": None,
        "max_width_m": point_summary.get("maximum_overall_width_m"),
        "notes": "Uses PCA projection from classified sidewalk points",
    },
    {
        "method": "Boundary-based",
        "average_width_m": boundary_summary.get("boundary_filtered_average_width_m"),
        "median_width_m": boundary_summary.get("boundary_filtered_median_width_m"),
        "max_width_m": boundary_summary.get("boundary_filtered_maximum_width_m"),
        "notes": "Uses separated kerb and HFE boundary files",
    },
])

comparison


## 11. Current Observations

### Point-based workflow
- Uses PCA projection
- More tolerant to incomplete boundaries
- Sensitive to classification spillover

### Boundary-based workflow
- Uses extracted sidewalk edges
- More geometrically stable
- Strongly dependent on boundary quality

### Validation observations
- Boundary filtering removes unrealistic pairings
- Boundary widths are now reasonably close to point-based estimates
- Remaining differences are likely caused by:
  - classification noise
  - curved geometry
  - intersections
  - local mismatch between opposite boundaries


## 12. Example Final Commands

### Point-based workflow

```bash
python3 -m metrics.width_metrics --input classified/utrecht_mlp_classified.laz --output outputs/width_metrics
```

### Boundary-based workflow

```bash
python3 -m metrics.width_metrics --kerb-file outputs/sidewalk_KI.laz --hfe-file outputs/sidewalk_HFE.laz --output outputs/width_metrics
```
